# 86 Billion Butterflies — Gut Microbiome Landscape Exploration

Interactive exploration of real gut microbiome profiles in reduced-dimensional space, with emphasis on mood/mental-state overlays.


In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
np.random.seed(42)

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

px.defaults.template = "plotly_dark"


In [ ]:
cwd = Path.cwd().resolve()
if (cwd / "analysis").exists() and (cwd / "src").exists():
    ROOT = cwd
elif cwd.name == "notebooks" and cwd.parent.name == "analysis":
    ROOT = cwd.parents[1]
else:
    ROOT = cwd

RAW_DIR = ROOT / "analysis" / "data" / "raw"
MICROBIOME_HD_DIR = RAW_DIR / "microbiome_hd" / "microbiomeHD"
MICROBIOME_HD_DATASETS_DIR = MICROBIOME_HD_DIR / "data" / "raw_otu_tables"
AGP_DIR = RAW_DIR / "american_gut"

TARGET_GENERA = [
    "Lactobacillus",
    "Bifidobacterium",
    "Enterococcus",
    "Streptococcus",
    "Bacteroides",
    "Faecalibacterium",
    "Roseburia",
    "Akkermansia",
    "Prevotella",
    "Clostridium",
    "Escherichia",
]

GENUS_SYNONYMS = {
    "Escherichia-Shigella": "Escherichia",
    "Shigella": "Escherichia",
}

print("ROOT:", ROOT)
print("MicrobiomeHD repo exists:", MICROBIOME_HD_DIR.exists())
print("MicrobiomeHD raw tables exist:", MICROBIOME_HD_DATASETS_DIR.exists())
print("AGP dir exists:", AGP_DIR.exists())


## Dataset checks

This notebook prioritizes real accessible datasets. MicrobiomeHD is the required source for this exploratory phase.


In [ ]:
dataset_status = pd.DataFrame(
    {
        "dataset": ["MicrobiomeHD raw OTU tables", "American Gut (optional local files)"],
        "available": [MICROBIOME_HD_DATASETS_DIR.exists(), AGP_DIR.exists()],
        "path": [str(MICROBIOME_HD_DATASETS_DIR), str(AGP_DIR)],
    }
)

display(dataset_status)
print("Target genera:", ", ".join(TARGET_GENERA))


## Optional loaders for real datasets

These run only if files are present. Otherwise the notebook continues in profile-exploration mode.


In [ ]:
def extract_genus(taxonomy_value):
    if pd.isna(taxonomy_value):
        return None
    text = str(taxonomy_value)
    parts = [p.strip() for p in text.split(";") if p.strip()]
    g = None
    for p in parts:
        if p.startswith("g__"):
            g = p.replace("g__", "").strip()
            break
    if g is None and len(parts) >= 6:
        g = parts[5].replace("g__", "").strip()
    if not g:
        g = parts[-1].replace("g__", "").strip() if parts else None
    if g in GENUS_SYNONYMS:
        g = GENUS_SYNONYMS[g]
    if isinstance(g, str) and "Clostridium" in g:
        g = "Clostridium"
    return g


def load_microbiomehd(raw_otu_tables_dir):
    if not raw_otu_tables_dir.exists():
        return pd.DataFrame()
    frames = []
    dataset_dirs = sorted([p for p in raw_otu_tables_dir.glob("*_results") if p.is_dir()])
    for dataset_dir in dataset_dirs:
        metadata_files = list(dataset_dir.glob("*.metadata.txt")) + list(dataset_dir.glob("*metadata*.txt"))
        otu_candidates = list(dataset_dir.glob("RDP/*.otu_table*.rdp_assigned"))
        if not otu_candidates:
            otu_candidates = list(dataset_dir.glob("**/*.otu_table*.rdp_assigned"))
        if not metadata_files or not otu_candidates:
            continue
        metadata_path = metadata_files[0]
        otu_path = otu_candidates[0]
        try:
            otu = pd.read_csv(otu_path, sep="\t", index_col=0, encoding="utf-8", encoding_errors="ignore")
            if otu.shape[1] == 0:
                continue
            otu_genus = otu.copy()
            otu_genus["genus"] = [extract_genus(idx) for idx in otu_genus.index]
            otu_genus = otu_genus.dropna(subset=["genus"])
            otu_genus = otu_genus.groupby("genus").sum(numeric_only=True)
            otu_rel = otu_genus.div(otu_genus.sum(axis=0).replace(0, np.nan), axis=1).fillna(0)
            otu_rel = otu_rel.reindex(TARGET_GENERA).fillna(0)
            sample_by_genus = otu_rel.T
            sample_by_genus.index.name = "sample_id"
            try:
                metadata = pd.read_csv(metadata_path, sep="\t", index_col=0, encoding="utf-8")
            except Exception:
                metadata = pd.read_csv(metadata_path, sep="\t", index_col=0, encoding="latin1", encoding_errors="ignore")
            merged = sample_by_genus.join(metadata, how="left")
            merged["dataset"] = dataset_dir.name
            merged["source"] = f"microbiomeHD_{dataset_dir.name}"
            merged["health_status"] = merged.get("DiseaseState", "unknown").astype(str)
            merged["sample_id"] = merged.index.astype(str)
            frames.append(merged)
        except Exception:
            continue
    if not frames:
        return pd.DataFrame()
    combined = pd.concat(frames, axis=0)
    for g in TARGET_GENERA:
        if g not in combined.columns:
            combined[g] = 0.0
    return combined


microbiomehd_df = load_microbiomehd(MICROBIOME_HD_DATASETS_DIR)
print("MicrobiomeHD loaded rows:", len(microbiomehd_df))
if len(microbiomehd_df):
    display(microbiomehd_df[["dataset", "health_status"]].head())


In [ ]:
def map_health_status(value):
    text = str(value).strip().lower()
    if text in {"h", "healthy", "control", "hc", "none", "nan", "unknown"}:
        return "healthy"
    if "mdd" in text or "depression" in text or "depressed" in text:
        return "MDD"
    if "anxiety" in text:
        return "anxiety"
    if "ibd" in text or "crohn" in text or "colitis" in text:
        return "IBD"
    if "obesity" in text or "obese" in text:
        return "obesity"
    if "crc" in text or "cancer" in text or "carcinoma" in text:
        return "CRC"
    if "cdi" in text or "clostridium difficile" in text:
        return "CDI"
    if "hiv" in text:
        return "HIV"
    if "t1d" in text or "type 1" in text or "diabetes" in text:
        return "T1D"
    if "aut" in text or "asd" in text or "autism" in text:
        return "ASD"
    if "par" in text or "parkinson" in text:
        return "Parkinsons"
    if "ra" in text or "rheumatoid" in text:
        return "RA"
    if "nash" in text or "fatty liver" in text:
        return "NASH"
    if "edd" in text or "decomp" in text or "encephalopathy" in text:
        return "EDD"
    if "mhe" in text:
        return "MHE"
    cleaned = text.replace(" ", "_")
    return cleaned[:30] if cleaned else "other_disease"


def load_agp_from_local(agp_dir):
    candidate_csvs = list(agp_dir.glob("*.csv")) + list(agp_dir.glob("*.tsv"))
    if not candidate_csvs:
        return pd.DataFrame()
    for path in candidate_csvs:
        try:
            sep = "\t" if path.suffix.lower() == ".tsv" else ","
            df = pd.read_csv(path, sep=sep)
            if set(TARGET_GENERA).issubset(set(df.columns)):
                out = df.copy()
                out["dataset"] = "american_gut"
                out["source"] = "agp"
                if "health_status" not in out.columns:
                    out["health_status"] = "unknown"
                if "sample_id" not in out.columns:
                    out["sample_id"] = [f"agp_{i:07d}" for i in range(len(out))]
                return out
        except Exception:
            continue
    return pd.DataFrame()


if len(microbiomehd_df) == 0:
    raise RuntimeError(
        "MicrobiomeHD data not found. Clone it to analysis/data/raw/microbiome_hd/microbiomeHD first."
    )

microbiomehd_df = microbiomehd_df.copy()
microbiomehd_df["health_status"] = microbiomehd_df["health_status"].apply(map_health_status)
microbiomehd_df["source"] = microbiomehd_df.get("source", "microbiomeHD")

agp_df = load_agp_from_local(AGP_DIR)
if len(agp_df):
    agp_df = agp_df.copy()
    agp_df["health_status"] = agp_df["health_status"].apply(map_health_status)
    agp_df["source"] = "agp"

analysis_df = pd.concat([microbiomehd_df, agp_df], axis=0, ignore_index=True) if len(agp_df) else microbiomehd_df.copy()

for g in TARGET_GENERA:
    analysis_df[g] = pd.to_numeric(analysis_df[g], errors="coerce").fillna(0)

analysis_df[TARGET_GENERA] = analysis_df[TARGET_GENERA].div(
    analysis_df[TARGET_GENERA].sum(axis=1).replace(0, np.nan),
    axis=0,
).fillna(0)

analysis_df = analysis_df[analysis_df[TARGET_GENERA].sum(axis=1) > 0].copy()

print("Total samples:", len(analysis_df))
print("By source:")
display(analysis_df["source"].value_counts().to_frame("n"))
print("By health_status:")
display(analysis_df["health_status"].value_counts().to_frame("n"))
analysis_df.shape


## CLR transform + PCA


In [ ]:
def clr_transform(frame, pseudocount=1e-6):
    x = frame.values + pseudocount
    log_x = np.log(x)
    gm = log_x.mean(axis=1, keepdims=True)
    return pd.DataFrame(log_x - gm, index=frame.index, columns=frame.columns)

X_raw = analysis_df[TARGET_GENERA].copy()
X_clr = clr_transform(X_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clr)

pca = PCA(n_components=min(len(TARGET_GENERA), len(X_scaled)))
X_pca = pca.fit_transform(X_scaled)

for i in range(min(6, X_pca.shape[1])):
    analysis_df[f"PC{i+1}"] = X_pca[:, i]

ev = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "explained_variance_ratio": pca.explained_variance_ratio_,
})
ev["cumulative"] = ev["explained_variance_ratio"].cumsum()
ev.head(8)


In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=ev["PC"], y=ev["explained_variance_ratio"], name="Explained variance"))
fig.add_trace(go.Scatter(x=ev["PC"], y=ev["cumulative"], mode="lines+markers", name="Cumulative"))
fig.update_layout(
    title="PCA Scree (interactive)",
    xaxis_title="Component",
    yaxis_title="Variance ratio",
    template="plotly_dark",
)
fig.show()


In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=TARGET_GENERA,
    columns=[f"PC{i+1}" for i in range(pca.components_.shape[0])],
)

loadings_top = loadings[[f"PC{i+1}" for i in range(min(5, loadings.shape[1]))]]
fig = px.imshow(
    loadings_top,
    color_continuous_scale="RdBu",
    zmin=-np.abs(loadings_top.values).max(),
    zmax=np.abs(loadings_top.values).max(),
    title="Genus loadings heatmap (PC1-PC5)",
    aspect="auto",
)
fig.update_layout(template="plotly_dark")
fig.show()


## Interactive landscape plots


In [ ]:
analysis_df["dominant_genus"] = analysis_df[TARGET_GENERA].idxmax(axis=1)
analysis_df["max_genus_abundance"] = analysis_df[TARGET_GENERA].max(axis=1)

analysis_df["butyrate_support"] = analysis_df[["Faecalibacterium", "Roseburia", "Akkermansia"]].sum(axis=1)
analysis_df["fermentative_support"] = analysis_df[["Lactobacillus", "Bifidobacterium"]].sum(axis=1)
analysis_df["inflammatory_pressure"] = analysis_df[["Escherichia", "Enterococcus", "Streptococcus"]].sum(axis=1)
analysis_df["prevotella_bacteroides_balance"] = analysis_df["Prevotella"] - analysis_df["Bacteroides"]
analysis_df["depressive_proxy_score"] = (
    analysis_df["inflammatory_pressure"]
    - analysis_df["butyrate_support"]
    - 0.5 * analysis_df["fermentative_support"]
)
analysis_df["anxiety_proxy_score"] = (
    analysis_df["inflammatory_pressure"]
    + 0.4 * analysis_df["prevotella_bacteroides_balance"].abs()
    - 0.6 * analysis_df["fermentative_support"]
)

dep_hi = analysis_df["depressive_proxy_score"].quantile(0.70)
dep_lo = analysis_df["depressive_proxy_score"].quantile(0.35)
anx_hi = analysis_df["anxiety_proxy_score"].quantile(0.70)
infl_hi = analysis_df["inflammatory_pressure"].quantile(0.70)
res_hi = analysis_df["butyrate_support"].quantile(0.70)

analysis_df["inferred_mental_state"] = "balanced_reference"
analysis_df.loc[analysis_df["depressive_proxy_score"] >= dep_hi, "inferred_mental_state"] = "depressive_like"
analysis_df.loc[analysis_df["anxiety_proxy_score"] >= anx_hi, "inferred_mental_state"] = "anxious_like"
analysis_df.loc[
    (analysis_df["inflammatory_pressure"] >= infl_hi) & (analysis_df["depressive_proxy_score"] >= dep_lo),
    "inferred_mental_state",
] = "inflammatory_distress"
analysis_df.loc[
    (analysis_df["butyrate_support"] >= res_hi) & (analysis_df["depressive_proxy_score"] < dep_lo),
    "inferred_mental_state",
] = "resilient_calm"

mood_state_map = {
    "healthy": "observed_healthy",
    "MDD": "observed_MDD",
    "anxiety": "observed_anxiety",
}
analysis_df["mood_state"] = analysis_df["health_status"].map(mood_state_map).fillna("inferred_from_microbiome")
analysis_df["mental_state_info"] = (
    "observed=" + analysis_df["health_status"].astype(str) + "; inferred=" + analysis_df["inferred_mental_state"].astype(str)
)

hover_cols = [
    "sample_id",
    "dataset",
    "source",
    "health_status",
    "mood_state",
    "inferred_mental_state",
    "mental_state_info",
    "depressive_proxy_score",
    "anxiety_proxy_score",
    "dominant_genus",
    "max_genus_abundance",
]

fig = px.scatter(
    analysis_df,
    x="PC1",
    y="PC2",
    color="health_status",
    symbol="source",
    hover_data=hover_cols,
    opacity=0.72,
    title="PC1 vs PC2 by observed health status",
)
fig.update_layout(template="plotly_dark")
fig.show()

fig = px.scatter(
    analysis_df,
    x="PC1",
    y="PC2",
    color="inferred_mental_state",
    symbol="source",
    hover_data=hover_cols,
    opacity=0.72,
    title="PC1 vs PC2 by inferred mental-state archetype",
)
fig.update_layout(template="plotly_dark")
fig.show()

fig = px.scatter_3d(
    analysis_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="inferred_mental_state",
    symbol="source",
    hover_data=hover_cols,
    opacity=0.68,
    title="3D PCA with observed + inferred mental-state popup",
)
fig.update_layout(template="plotly_dark")
fig.show()


In [ ]:
if UMAP_AVAILABLE:
    reducer2d = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.25, random_state=42)
    umap2d = reducer2d.fit_transform(X_scaled)
    analysis_df["UMAP1"] = umap2d[:, 0]
    analysis_df["UMAP2"] = umap2d[:, 1]

    fig = px.scatter(
        analysis_df,
        x="UMAP1",
        y="UMAP2",
        color="health_status",
        symbol="source",
        hover_data=hover_cols,
        opacity=0.72,
        title="2D UMAP by health_status",
    )
    fig.update_layout(template="plotly_dark")
    fig.show()

    fig = px.scatter(
        analysis_df,
        x="UMAP1",
        y="UMAP2",
        color="dominant_genus",
        symbol="source",
        hover_data=hover_cols,
        opacity=0.72,
        title="2D UMAP by dominant genus",
    )
    fig.update_layout(template="plotly_dark")
    fig.show()

    reducer3d = umap.UMAP(n_components=3, n_neighbors=30, min_dist=0.25, random_state=42)
    umap3d = reducer3d.fit_transform(X_scaled)
    analysis_df["UMAP3_1"] = umap3d[:, 0]
    analysis_df["UMAP3_2"] = umap3d[:, 1]
    analysis_df["UMAP3_3"] = umap3d[:, 2]

    fig = px.scatter_3d(
        analysis_df,
        x="UMAP3_1",
        y="UMAP3_2",
        z="UMAP3_3",
        color="health_status",
        symbol="source",
        hover_data=hover_cols,
        opacity=0.65,
        title="3D UMAP exploration",
    )
    fig.update_layout(template="plotly_dark")
    fig.show()
else:
    print("umap-learn not available. Install umap-learn to enable UMAP plots.")


## Meaning Layer: Latent Clusters and Mental-State Interpretation


In [ ]:
latent_cols = [c for c in ["PC1", "PC2", "PC3", "PC4", "PC5"] if c in analysis_df.columns]
latent_base = analysis_df[latent_cols].values
proxy_cols = [
    "depressive_proxy_score",
    "anxiety_proxy_score",
    "butyrate_support",
    "inflammatory_pressure",
]
latent_proxy = analysis_df[proxy_cols].values
latent_proxy_scaled = StandardScaler().fit_transform(latent_proxy)
latent_matrix = np.hstack([latent_base, latent_proxy_scaled])

min_hotspots = 6
max_hotspots = 12
candidate_k = [k for k in range(min_hotspots, max_hotspots + 1) if k < len(analysis_df)]
if len(candidate_k) == 0:
    candidate_k = [min(4, max(2, len(analysis_df) - 1))]

k_scores = []
for k in candidate_k:
    model = KMeans(n_clusters=k, random_state=42, n_init=30)
    labels = model.fit_predict(latent_matrix)
    score = silhouette_score(latent_matrix, labels) if len(np.unique(labels)) > 1 else -1
    k_scores.append((k, score))

best_k = sorted(k_scores, key=lambda x: x[1], reverse=True)[0][0]
cluster_model = KMeans(n_clusters=best_k, random_state=42, n_init=40)
analysis_df["latent_cluster"] = cluster_model.fit_predict(latent_matrix).astype(int)

if UMAP_AVAILABLE:
    latent_reducer_3d = umap.UMAP(n_components=3, n_neighbors=50, min_dist=0.15, random_state=42)
    latent_3d = latent_reducer_3d.fit_transform(latent_matrix)
    analysis_df["LATENT3D_1"] = latent_3d[:, 0]
    analysis_df["LATENT3D_2"] = latent_3d[:, 1]
    analysis_df["LATENT3D_3"] = latent_3d[:, 2]
else:
    latent_pca_3d = PCA(n_components=3, random_state=42)
    latent_3d = latent_pca_3d.fit_transform(latent_matrix)
    analysis_df["LATENT3D_1"] = latent_3d[:, 0]
    analysis_df["LATENT3D_2"] = latent_3d[:, 1]
    analysis_df["LATENT3D_3"] = latent_3d[:, 2]

cluster_rows = []
global_genus_mean = analysis_df[TARGET_GENERA].mean()

for cluster_id, grp in analysis_df.groupby("latent_cluster"):
    observed_dist = grp["health_status"].astype(str).value_counts(normalize=True)
    inferred_dist = grp["inferred_mental_state"].astype(str).value_counts(normalize=True)
    dominant_health = observed_dist.index[0] if len(observed_dist) else "unknown"
    dominant_inferred = inferred_dist.index[0] if len(inferred_dist) else "unknown"
    observed_profile = " | ".join([f"{k}:{v:.2f}" for k, v in observed_dist.head(4).items()])
    inferred_profile = " | ".join([f"{k}:{v:.2f}" for k, v in inferred_dist.head(4).items()])
    delta = grp[TARGET_GENERA].mean() - global_genus_mean
    top_positive = delta.sort_values(ascending=False).head(3).index.tolist()
    top_signature = ", ".join(top_positive)
    if dominant_inferred == "resilient_calm":
        av_hint = "open_breathing_luminous"
    elif dominant_inferred == "depressive_like":
        av_hint = "low_energy_slow_dense"
    elif dominant_inferred == "anxious_like":
        av_hint = "high_frequency_restless"
    elif dominant_inferred == "inflammatory_distress":
        av_hint = "tense_granular_contrast"
    else:
        av_hint = "balanced_transitional"
    cluster_rows.append(
        {
            "latent_cluster": int(cluster_id),
            "n_samples": int(len(grp)),
            "dominant_health": dominant_health,
            "dominant_inferred_state": dominant_inferred,
            "observed_profile": observed_profile,
            "inferred_profile": inferred_profile,
            "healthy_fraction": float(observed_dist.get("healthy", 0.0)),
            "mdd_fraction": float(observed_dist.get("MDD", 0.0)),
            "anxiety_fraction": float(observed_dist.get("anxiety", 0.0)),
            "top_signature_genera": top_signature,
            "av_preset_hint": av_hint,
        }
    )

cluster_summary = pd.DataFrame(cluster_rows).sort_values("n_samples", ascending=False)
analysis_df = analysis_df.merge(cluster_summary, on="latent_cluster", how="left")
analysis_df["mental_state_popup"] = (
    "observed{" + analysis_df["observed_profile"].astype(str) + "} ; inferred{" + analysis_df["inferred_profile"].astype(str) + "}"
)
analysis_df["cluster_label"] = analysis_df.apply(
    lambda r: f"C{int(r['latent_cluster'])}::{r['dominant_inferred_state']}", axis=1
)

display(pd.DataFrame(k_scores, columns=["k", "silhouette"]).sort_values("silhouette", ascending=False))
display(cluster_summary)


In [ ]:
meaning_hover_cols = [
    "sample_id",
    "dataset",
    "health_status",
    "inferred_mental_state",
    "cluster_label",
    "mental_state_popup",
    "top_signature_genera",
    "av_preset_hint",
    "depressive_proxy_score",
    "anxiety_proxy_score",
    "dominant_genus",
    "max_genus_abundance",
]

fig = px.scatter_3d(
    analysis_df,
    x="LATENT3D_1",
    y="LATENT3D_2",
    z="LATENT3D_3",
    color="cluster_label",
    hover_data=meaning_hover_cols,
    opacity=0.72,
    title="3D latent microbiome space with mental-state popup",
)
fig.update_layout(
    template="plotly_dark",
    showlegend=False,
)
fig.show()


## Hotspot Centroids (Dedicated View)

This separate view isolates hotspot centroids so labels are easier to read and compare.


In [ ]:
cluster_centroids = (
    analysis_df.groupby("latent_cluster")[["LATENT3D_1", "LATENT3D_2", "LATENT3D_3"]]
    .mean()
    .reset_index()
    .merge(cluster_summary, on="latent_cluster", how="left")
)
cluster_centroids["cluster_label"] = cluster_centroids.apply(
    lambda r: f"C{int(r['latent_cluster'])}::{r['dominant_inferred_state']}", axis=1
)
cluster_centroids["mental_state_popup"] = cluster_centroids.apply(
    lambda r: (
        f"quality={r['dominant_inferred_state']} | "
        f"healthy={r['healthy_fraction']:.2f}, MDD={r['mdd_fraction']:.2f}, anxiety={r['anxiety_fraction']:.2f}"
    ),
    axis=1,
)

fig = px.scatter_3d(
    cluster_centroids,
    x="LATENT3D_1",
    y="LATENT3D_2",
    z="LATENT3D_3",
    color="cluster_label",
    size="n_samples",
    hover_name="cluster_label",
    hover_data=["mental_state_popup", "n_samples"],
    title="Hotspot centroids in latent 3D space",
)
fig.update_layout(template="plotly_dark", legend=dict(itemsizing="constant"))
fig.show()


## Mental-State Qualities Per Hotspot

These labels focus on mental-state qualities (arousal, stability, tension, resilience, affective valence), not disease names.

The inference combines:
- observed profile mix where available (`MDD`, `anxiety`, `healthy`), and
- microbiome-based proxies tied to literature patterns (butyrate support, inflammatory pressure, fermentative support).


In [ ]:
from IPython.display import Markdown, display

summary = cluster_summary.copy()
summary = summary.sort_values("n_samples", ascending=False).reset_index(drop=True)

cluster_proxy = (
    analysis_df.groupby("latent_cluster")[[
        "depressive_proxy_score",
        "anxiety_proxy_score",
        "butyrate_support",
        "inflammatory_pressure",
        "fermentative_support",
    ]]
    .mean()
    .reset_index()
)
summary = summary.merge(cluster_proxy, on="latent_cluster", how="left")

def z_series(s):
    sd = s.std(ddof=0)
    if sd == 0 or pd.isna(sd):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / sd

summary["z_dep"] = z_series(summary["depressive_proxy_score"])
summary["z_anx"] = z_series(summary["anxiety_proxy_score"])
summary["z_but"] = z_series(summary["butyrate_support"])
summary["z_inf"] = z_series(summary["inflammatory_pressure"])
summary["z_ferm"] = z_series(summary["fermentative_support"])

def infer_quality(row):
    if row["z_but"] >= 0.6 and row["z_inf"] <= -0.2 and row["z_dep"] <= -0.2:
        return "resilient / calm / grounded"
    if row["z_dep"] >= 0.8 and row["z_inf"] >= 0.2:
        return "low-valence / heavy / depressive-like"
    if row["z_anx"] >= 0.8 and row["z_inf"] >= 0.2:
        return "high-arousal / tense / anxious-like"
    if row["z_inf"] >= 1.0 and row["z_dep"] >= 0.2:
        return "stress-reactive / dysregulated"
    if row["z_ferm"] >= 0.6 and row["z_dep"] <= 0.2:
        return "socially-open / adaptive / stable"
    return "mixed / transitional"

summary["mental_quality"] = summary.apply(infer_quality, axis=1)

lines = ["### Hotspot Mental-State Inference"]
for _, r in summary.sort_values("latent_cluster").iterrows():
    cid = int(r["latent_cluster"])
    lines.append(
        f"- **C{cid}**: {r['mental_quality']}  "
        f"(observed mix: {r['observed_profile']}; inferred mix: {r['inferred_profile']}; "
        f"signature genera: {r['top_signature_genera']}; A/V hint: {r['av_preset_hint']})"
    )

display(Markdown("\n".join(lines)))

display(
    summary[
        [
            "latent_cluster",
            "n_samples",
            "mental_quality",
            "dominant_inferred_state",
            "top_signature_genera",
            "depressive_proxy_score",
            "anxiety_proxy_score",
            "butyrate_support",
            "inflammatory_pressure",
            "fermentative_support",
            "av_preset_hint",
        ]
    ].sort_values("latent_cluster")
)
